In [ ]:
# ==============================================================================
# MỤC 1: SỬ DỤNG THƯ VIỆN (SCIKIT-LEARN) CHO THUẬT TOÁN NAIVE BAYES
# ==============================================================================

# Import thư viện numpy để xử lý mảng số học và ma trận
import numpy as np
# Import GaussianNB từ thư viện scikit-learn (phù hợp cho dữ liệu đặc trưng liên tục)
from sklearn.naive_bayes import GaussianNB
# Import hàm chia dữ liệu thành tập huấn luyện (train) và tập kiểm thử (test)
from sklearn.model_selection import train_test_split
# Import các độ đo đánh giá hiệu năng mô hình phân loại
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def train_and_predict_sklearn(X_train, y_train, X_test):
    # Khởi tạo mô hình Gaussian Naive Bayes từ thư viện sklearn
    model = GaussianNB()
    
    # Huấn luyện mô hình trên tập dữ liệu huấn luyện (tính toán mean, variance và prior cho từng lớp)
    model.fit(X_train, y_train)
    
    # Dự đoán nhãn lớp cho tập dữ liệu kiểm thử
    y_pred = model.predict(X_test)
    
    # Dự đoán xác suất rơi vào từng lớp của mỗi mẫu dữ liệu kiểm thử
    y_proba = model.predict_proba(X_test)
    
    # Trả về mô hình đã huấn luyện, nhãn dự đoán và xác suất dự đoán
    return model, y_pred, y_proba


In [ ]:
# ==============================================================================
# MỤC 2: TỰ VIẾT CODE THUẬT TOÁN GAUSSIAN NAIVE BAYES TỪ ĐẦU (FROM SCRATCH)
# ==============================================================================

class CustomGaussianNB:
    def __init__(self):
        # Khởi tạo các biến lưu trữ tham số sau khi huấn luyện:
        # classes: danh sách các nhãn lớp duy nhất
        # mean: giá trị trung bình của từng đặc trưng theo mỗi lớp
        # var: phương sai của từng đặc trưng theo mỗi lớp
        # priors: xác suất tiên nghiệm P(y) của mỗi lớp
        self.classes = None
        self.mean = None
        self.var = None
        self.priors = None

    def fit(self, X, y):
        # Lấy số lượng mẫu (n_samples) và số lượng đặc trưng (n_features)
        n_samples, n_features = X.shape
        # Tìm tất cả các nhãn lớp duy nhất có trong tập nhãn y
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        # Khởi tạo ma trận lưu trung bình, phương sai (kích thước: số lớp x số đặc trưng)
        # và mảng lưu xác suất tiên nghiệm (kích thước: số lớp)
        self.mean = np.zeros((n_classes, n_features), dtype=np.float64)
        self.var = np.zeros((n_classes, n_features), dtype=np.float64)
        self.priors = np.zeros(n_classes, dtype=np.float64)

        # Duyệt qua từng lớp để tính các tham số thống kê
        for idx, c in enumerate(self.classes):
            # Lọc ra tất cả các mẫu dữ liệu thuộc về lớp c
            X_c = X[y == c]
            # Tính giá trị trung bình theo từng cột đặc trưng của lớp c
            self.mean[idx, :] = X_c.mean(axis=0)
            # Tính phương sai theo từng cột đặc trưng của lớp c (cộng thêm 1e-9 để tránh lỗi chia cho 0)
            self.var[idx, :] = X_c.var(axis=0) + 1e-9
            # Tính xác suất tiên nghiệm P(c) = số mẫu lớp c / tổng số mẫu
            self.priors[idx] = X_c.shape[0] / float(n_samples)

    def _pdf(self, class_idx, x):
        # Tính hàm mật độ xác suất phân phối chuẩn (Gaussian PDF) cho vector đặc trưng x tại lớp class_idx
        mean = self.mean[class_idx]
        var = self.var[class_idx]
        # Tử số: exp(-(x - mean)^2 / (2 * var))
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))
        # Mẫu số: sqrt(2 * pi * var)
        denominator = np.sqrt(2 * np.pi * var)
        # Trả về mảng xác suất có điều kiện P(x_i | y) cho từng đặc trưng
        return numerator / denominator

    def _predict_single(self, x):
        # Danh sách lưu xác suất hậu nghiệm (log-posterior) của mẫu x cho từng lớp
        posteriors = []

        # Tính xác suất hậu nghiệm cho từng lớp theo định lý Bayes
        for idx, c in enumerate(self.classes):
            # Sử dụng logarit để biến phép nhân xác suất thành phép cộng, tránh tràn số dưới (underflow)
            prior = np.log(self.priors[idx])
            # Tổng log của các xác suất có điều kiện P(x_i | y) dựa trên giả định độc lập có điều kiện
            class_conditional = np.sum(np.log(self._pdf(idx, x) + 1e-12))
            # Log của xác suất hậu nghiệm: log(P(y)) + sum(log(P(x_i | y)))
            posterior = prior + class_conditional
            posteriors.append(posterior)

        # Chọn và trả về nhãn lớp có xác suất hậu nghiệm lớn nhất (Maximum A Posteriori - MAP)
        return self.classes[np.argmax(posteriors)]

    def predict(self, X):
        # Áp dụng hàm dự đoán cho từng mẫu trong tập dữ liệu X và trả về mảng kết quả
        y_pred = [self._predict_single(x) for x in X]
        return np.array(y_pred)


In [ ]:
# ==============================================================================
# MỤC 3: TẠO DATA MOCK VÀ KIỂM THỬ SO SÁNH 2 THUẬT TOÁN (THƯ VIỆN VS TỰ VIẾT)
# ==============================================================================

# Import hàm make_classification để tạo bộ dữ liệu giả lập (mock data) cho bài toán phân loại
from sklearn.datasets import make_classification

# Bước 1: Tạo dữ liệu giả lập (Mock Data)
# Tạo 500 mẫu dữ liệu, 4 đặc trưng, chia thành 3 lớp phân loại, cố định random_state=42 để tái lập kết quả
X, y = make_classification(
    n_samples=500,
    n_features=4,
    n_informative=4,
    n_redundant=0,
    n_classes=3,
    n_clusters_per_class=1,
    random_state=42
)

# Chia dữ liệu thành 80% tập huấn luyện (400 mẫu) và 20% tập kiểm thử (100 mẫu)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Kích thước tập huấn luyện: {X_train.shape}, Tập kiểm thử: {X_test.shape}\n")

# Bước 2: Chạy thử nghiệm với mô hình từ thư viện Scikit-Learn (Mục 1)
sklearn_model, y_pred_sklearn, y_proba_sklearn = train_and_predict_sklearn(X_train, y_train, X_test)
acc_sklearn = accuracy_score(y_test, y_pred_sklearn)

# Bước 3: Chạy thử nghiệm với mô hình tự viết CustomGaussianNB (Mục 2)
custom_model = CustomGaussianNB()
custom_model.fit(X_train, y_train)
y_pred_custom = custom_model.predict(X_test)
acc_custom = accuracy_score(y_test, y_pred_custom)

# Bước 4: In kết quả so sánh giữa 2 phương pháp
print("=== KẾT QUẢ ĐÁNH GIÁ & SO SÁNH ===")
print(f"Độ chính xác (Accuracy) - Thư viện Sklearn : {acc_sklearn * 100:.2f}%")
print(f"Độ chính xác (Accuracy) - Thuật toán tự viết: {acc_custom * 100:.2f}%")

# Kiểm tra tỷ lệ trùng khớp dự đoán giữa thuật toán tự viết và thư viện chuẩn
match_rate = np.mean(y_pred_sklearn == y_pred_custom) * 100
print(f"Tỷ lệ dự đoán trùng khớp giữa 2 mô hình      : {match_rate:.2f}%\n")

# In ma trận nhầm lẫn (Confusion Matrix) của mô hình tự viết
print("Ma trận nhầm lẫn (Confusion Matrix - Custom Model):")
print(confusion_matrix(y_test, y_pred_custom))
print("\nBáo cáo phân loại chi tiết (Classification Report - Custom Model):")
print(classification_report(y_test, y_pred_custom))

# In thử 5 mẫu dự đoán đầu tiên để đối chiếu trực tiếp
print("Đối chiếu 5 mẫu đầu tiên trên tập test (Thực tế | Sklearn | Tự viết):")
for i in range(5):
    print(f"Mẫu {i+1}: Nhãn thực tế = {y_test[i]} | Sklearn = {y_pred_sklearn[i]} | Tự viết = {y_pred_custom[i]}")
